In [1]:
!pip install -q pypdf sentence-transformers chromadb groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.2/382.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [2]:
import os
import re
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq

In [3]:
from google.colab import files

uploaded = files.upload()

PDF_PATH = list(uploaded.keys())[0]

print("Uploaded PDF:", PDF_PATH)

Saving HandBook_b9c6eSTUDENT HANDBOOK 2025.pdf to HandBook_b9c6eSTUDENT HANDBOOK 2025.pdf
Uploaded PDF: HandBook_b9c6eSTUDENT HANDBOOK 2025.pdf


In [4]:
reader = PdfReader(PDF_PATH)

pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()

    if text:
        text = re.sub(r"\s+", " ", text).strip()

        pages.append({
            "page": page_number,
            "text": text
        })

print("Total pages extracted:", len(pages))

print("\nSample text:\n")
print(pages[0]["text"][:1000])

Total pages extracted: 69

Sample text:

STUDENTHANDBOOK (For Guidance & Compliance by Students) 2025-2026


In [5]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []

    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size

        chunk = text[start:end]

        # Try to end at a sentence boundary
        if end < text_length:
            last_period = chunk.rfind(". ")

            if last_period > chunk_size * 0.5:
                chunk = chunk[:last_period + 1]
                end = start + last_period + 1

        chunks.append(chunk.strip())

        start = end - overlap

        if start < 0:
            start = 0

        if end >= text_length:
            break

    return chunks

In [6]:
all_chunks = []

chunk_id = 0

for page_data in pages:

    page_number = page_data["page"]
    text = page_data["text"]

    chunks = chunk_text(
        text,
        chunk_size=1000,
        overlap=200
    )

    for chunk in chunks:

        if len(chunk.strip()) > 50:

            all_chunks.append({
                "id": f"chunk_{chunk_id}",
                "text": chunk,
                "page": page_number
            })

            chunk_id += 1


print("Total chunks created:", len(all_chunks))

print("\nSample Chunk:")
print(all_chunks[0])

Total chunks created: 190

Sample Chunk:
{'id': 'chunk_0', 'text': 'STUDENTHANDBOOK (For Guidance & Compliance by Students) 2025-2026', 'page': 1}


In [7]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [8]:
chunk_texts = [
    chunk["text"]
    for chunk in all_chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", len(embeddings[0]))

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Number of embeddings: 190
Embedding dimension: 384


In [9]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="student_handbook"
)

print("Vector database created.")

Vector database created.


In [10]:
collection.add(
    ids=[
        chunk["id"]
        for chunk in all_chunks
    ],

    documents=[
        chunk["text"]
        for chunk in all_chunks
    ],

    embeddings=embeddings.tolist(),

    metadatas=[
        {
            "page": chunk["page"]
        }
        for chunk in all_chunks
    ]
)

print("Chunks successfully stored in ChromaDB.")

Chunks successfully stored in ChromaDB.


In [12]:
from getpass import getpass

GROQ_API_KEY = getpass("Enter your Groq API Key: ")

client = Groq(
    api_key=GROQ_API_KEY
)

Enter your Groq API Key: ··········


In [13]:
def retrieve_context(question, top_k=4):

    question_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    retrieved_chunks = []

    for i in range(len(results["documents"][0])):

        retrieved_chunks.append({
            "text": results["documents"][0][i],
            "page": results["metadatas"][0][i]["page"],
            "distance": results["distances"][0][i]
        })

    return retrieved_chunks

In [14]:
test_question = "What is the minimum attendance requirement for appearing in the end-semester examination?"

retrieved = retrieve_context(
    test_question,
    top_k=4
)

print("QUESTION:")
print(test_question)

print("\n" + "=" * 80)

for i, item in enumerate(retrieved, start=1):

    print(f"\nRETRIEVED CHUNK {i}")
    print(f"PAGE: {item['page']}")
    print("-" * 50)
    print(item["text"][:1000])

QUESTION:
What is the minimum attendance requirement for appearing in the end-semester examination?


RETRIEVED CHUNK 1
PAGE: 19
--------------------------------------------------
by the Hon’ble Vice Chancellor by condoning attendance up to 5% in a course with B+ cap on Grade. • However, Attendance Rules of relevant Regulatory Bodies/AUUP will be applicable whichever is higher. • Minimum Attendance criteria applicable on the programmes under Regulatory bodies is as under: Regulatory Body Minimum Attendance Criteria to appear in the End Semester Examination RCI 80% Course Work, 90% for Practicum & 100% in School Internship PCI 80% NCTE 80% Normal & 90% in School Internship BCI 75% CoA 75% subject to student is passed in Continuous Internal Assessment IAP (Physiotherapy Association) 75% AIOTA (Occupational Therapy Association) 75%

RETRIEVED CHUNK 2
PAGE: 40
--------------------------------------------------
are conducted prior to commencement of the next academic session, so that studen

In [15]:
def generate_answer(question, retrieved_chunks):

    context = ""

    for i, item in enumerate(retrieved_chunks, start=1):

        context += f"""
[Context {i} | Page {item['page']}]

{item['text']}

"""

    prompt = f"""
You are a university Student Handbook assistant.

Answer the student's question using ONLY the information provided
in the retrieved context below.

Rules:
1. Do not use outside knowledge.
2. Do not make up information.
3. If the answer is not clearly available in the context, say:
   "The information is not available in the retrieved sections of the Student Handbook."
4. Give a clear and concise answer.
5. When possible, mention the relevant page number.

Retrieved Context:
{context}

Student Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer questions strictly using the provided "
                    "Student Handbook context."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,
        max_tokens=500
    )

    return response.choices[0].message.content

In [16]:
def ask_student_handbook(question, top_k=4):

    # Step 1: Retrieve relevant chunks
    retrieved_chunks = retrieve_context(
        question,
        top_k=top_k
    )

    # Step 2: Generate answer using Groq
    answer = generate_answer(
        question,
        retrieved_chunks
    )

    # Get unique source pages
    source_pages = sorted(
        list(
            set(
                item["page"]
                for item in retrieved_chunks
            )
        )
    )

    # Display output
    print("=" * 80)

    print("\nSTUDENT QUESTION")
    print(question)

    print("\n" + "↓" * 40)

    print("\nRETRIEVED CONTEXT\n")

    for i, item in enumerate(retrieved_chunks, start=1):

        print(f"[Context {i} | Page {item['page']}]")
        print(item["text"])
        print("\n" + "-" * 80 + "\n")

    print("GENERATED ANSWER")
    print(answer)

    print("\n" + "↓" * 40)

    print("\nSOURCE / PAGE NUMBER")
    print("Relevant pages:", source_pages)

    print("\n" + "=" * 80)

    return {
        "question": question,
        "retrieved_context": retrieved_chunks,
        "answer": answer,
        "source_pages": source_pages
    }

In [18]:
result_1 = ask_student_handbook(
    "What is the minimum attendance requirement for appearing in the end-semester examination?"
)


STUDENT QUESTION
What is the minimum attendance requirement for appearing in the end-semester examination?

↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓

RETRIEVED CONTEXT

[Context 1 | Page 19]
by the Hon’ble Vice Chancellor by condoning attendance up to 5% in a course with B+ cap on Grade. • However, Attendance Rules of relevant Regulatory Bodies/AUUP will be applicable whichever is higher. • Minimum Attendance criteria applicable on the programmes under Regulatory bodies is as under: Regulatory Body Minimum Attendance Criteria to appear in the End Semester Examination RCI 80% Course Work, 90% for Practicum & 100% in School Internship PCI 80% NCTE 80% Normal & 90% in School Internship BCI 75% CoA 75% subject to student is passed in Continuous Internal Assessment IAP (Physiotherapy Association) 75% AIOTA (Occupational Therapy Association) 75%

--------------------------------------------------------------------------------

[Context 2 | Page 40]
are conducted prior to commencement of the

In [19]:
result_2 = ask_student_handbook(
    "How many days of academic leave can a student apply for?"
)


STUDENT QUESTION
How many days of academic leave can a student apply for?

↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓

RETRIEVED CONTEXT

[Context 1 | Page 37]
attend all classes including practical with junior batch and appear in both Continuous Internal Assessment as well as End Term Examination of that year. – Pay the prescribed Academic Fee of the Year. – Any exams given by the student in the previous year(s) for the repeat year will be treated as null & void. – Such Students will be treated as student of one-year lower batch. However, the enrolment number will continue to remain the same. The students will attend the classes of all courses as prescribed for lower batch. B) Academic Break • Students who apply for Academic Break and the case is recommended by the Head of Institutions for justifiable reasons to be recorded, can be granted Academic Break of one year for programmes of two years duration and two Academic Breaks of one year each for programmes of three years and above dur

In [20]:
result_3 = ask_student_handbook(
    "What are the rules for examination misconduct?"
)


STUDENT QUESTION
What are the rules for examination misconduct?

↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓

RETRIEVED CONTEXT

[Context 1 | Page 25]
any book, notes, paper or any other material or device or from any other student, to do any of these things or facilitating or rendering any assistance to any other student to do any of these things. • Arranging to impersonate any person, whosoever he may be, or for himself or impersonating for the other student at the examination. • Forging a document or using a forged document knowing it to be forged in any manner relating to the examination. • Any other act of omission or commission declared by the Academic Council/Executive Council to be unfair in respect of any or all examinations. 3.11.3 Penalties for Indiscipline • The Examination Discipline Committee may recommend penalties as per Para 30(8) of R01, Regulation on conduct of Examination, Scheme of Evaluation and Discipline among Students in the Examination. • The entire online exami

In [21]:
result_4 = ask_student_handbook(
    "What are the library timings?"
)


STUDENT QUESTION
What are the library timings?

↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓

RETRIEVED CONTEXT

[Context 1 | Page 15]
13 2.19 ACADEMIC CALENDAR Every year Academic Activity commences with the planning of Block Academic Calendar. Detailed Annual Academic Calendar at Institution/Programme level is aligned with University Block Academic Calendar and mentions the dates scheduled for following events/ activities/meetings: Academic/Examination Related • Registration/Re-registration • APW Submission • GSSC Milestones • Last Teaching Day • Date of Important Submissions • NTCC Calendar • Ph.D Calendar • Examination Calendar • Autumn Break/Winter Break Meetings • CR Meetings • Club-Committee Meetings • Mentor-Mentee Meetings • IQAC Meetings • Faculty Meetings Events • Sangathan • University Fresher’s Party • Scholarship Award Function • Amity Youth Festival • Conference/Workshops and other Events Calendar • Convocation Guest Lectures/Training and Workshops • Student Development Pro

In [22]:
result_5 = ask_student_handbook(
    "What disciplinary action can be taken against a student for violating university rules?"
)


STUDENT QUESTION
What disciplinary action can be taken against a student for violating university rules?

↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓

RETRIEVED CONTEXT

[Context 1 | Page 26]
Council on the recommendations of the Vice Chancellor, on the expiry of three years or such period as specified after such expulsion exempt a student from further operation of the punishment awarded. c) If within four months from the publication of the results, it is brought to the notice of the Controller of Examinations that a student was guilty of the use of dishonest or unfair means at the examination in respect of which his/her result was declared, the provisions of these regulations shall apply mutatis mutandis to the case of such a student provided that before imposing any penalty including the penalty of cancellation of his/her result, he/she shall be given another opportunity to show cause against the proposed punishment and his/her explanation, if any, shall be considered by the Academic C

In [23]:
results = [
    result_1,
    result_2,
    result_3,
    result_4,
    result_5
]

print("\nRAG SYSTEM VERIFICATION\n")

for i, result in enumerate(results, start=1):

    print("=" * 80)

    print(f"QUESTION {i}:")
    print(result["question"])

    print("\nGENERATED ANSWER:")
    print(result["answer"])

    print("\nSOURCE PAGES:")
    print(result["source_pages"])

    print("\nVERIFICATION:")
    print("Check whether the answer is supported by the retrieved context and source pages.")

    print("=" * 80)


RAG SYSTEM VERIFICATION

QUESTION 1:
What is the minimum attendance requirement for appearing in the end-semester examination?

GENERATED ANSWER:
The minimum attendance requirement for appearing in the end-semester examination varies depending on the Regulatory Body. As per Page 19, the minimum attendance criteria are as follows: 
- RCI: 80% for Course Work, 90% for Practicum, and 100% for School Internship
- PCI: 80%
- NCTE: 80% (normal) and 90% (in School Internship)
- BCI: 75%
- CoA: 75% (subject to passing in Continuous Internal Assessment)
- IAP (Physiotherapy Association): 75%
- AIOTA (Occupational Therapy Association): 75%
However, a student whose attendance is less than 75% will not be permitted to appear in the End Semester Examination unless permitted by the Hon'ble Vice Chancellor under extreme and special circumstances.

SOURCE PAGES:
[19, 20, 40]

VERIFICATION:
Check whether the answer is supported by the retrieved context and source pages.
QUESTION 2:
How many days of ac

In [25]:
# =====================================================
# INTERACTIVE STUDENT HANDBOOK RAG ASSISTANT
# =====================================================

while True:

    question = input("\nAsk a question about the Student Handbook\n(or type 'exit' to stop):\n\n")

    # Exit condition
    if question.lower() in ["exit", "quit", "stop"]:
        print("\nRAG Assistant stopped.")
        break

    # Ignore empty questions
    if not question.strip():
        print("Please enter a valid question.")
        continue

    # Run RAG pipeline
    ask_student_handbook(
        question,
        top_k=4
    )


Ask a question about the Student Handbook
(or type 'exit' to stop):

How many canteens are there?

STUDENT QUESTION
How many canteens are there?

↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓

RETRIEVED CONTEXT

[Context 1 | Page 55]
53 • For any complaint or suggestion, hostel residents can inform the Cafeteria Management Committee in writing with a copy to Director Hostels and DSW. • Other facilities include Daily-needs Store, a Saloon, a florist, a bookshop and a stationery shop which provides photocopying, scanning, printing and binding facilities. The shop also undertakes courier services, recharge of mobile phones and ticketing. In addition, there is a food court which houses outlets of Dominos, Mama Mia, Bikanerwala, Mrs. Kaur's Cookies, Amul Milk, Parlor, Café Coffee Day, Dosa Plaza & Subway. 5.10.12 Clubs & Committees Student Clubs are excellent platforms for exploring the inner potential of the students as per their interests. There are two types of clubs – • University Level Clu